In [3]:
import h5py,os,re,click
import numpy as np

ens='cB211.072.64'

path=f'/p/project1/ngff/li47/code/glwc3/project2/05_moments/dataPrepare/{ens}/data_aux/cfgs_run'
with open(path,'r') as f:
    cfgs=f.read().splitlines()
cfg2old=lambda cfg: cfg[1:]+'_r'+{'a':'0','b':'1','c':'2','d':'3'}[cfg[0]]
cfg2new=lambda cfg: {'0':'a','1':'b','2':'c','3':'d'}[cfg[-1]] + cfg[:4]

path=f'/p/project1/ngff/li47/code/projectData/07_Nsgm/charges/{ens}.h5'
with h5py.File(path,'w') as fw:
    fw.create_dataset('cfgs',data=cfgs)
    
    path='/p/project1/ngff/li47/code/projectData/07_Nsgm/fromChristos/charges_B64.h5'
    with h5py.File(path) as f:
    #     for key in f.keys():
    #         if not key.endswith('_twop'):
    #             continue
    #         tf=int(key.split('_')[0][2:])
    #         t=np.array([f[key][cfg2old(cfg)][()] for cfg in cfgs])
    #         fw.create_dataset(f'conn_c2pt/{tf}',data=t)
        
        for g in ['gS','gA','gT']:
            for tf_str in f[f'{g}/up'].keys():
                tf=int(tf_str[2:])
                tu=np.array([f[f'{g}/up/{tf_str}/{cfg2old(cfg)}'][:] for cfg in cfgs])
                td=np.array([f[f'{g}/dn/{tf_str}/{cfg2old(cfg)}'][:] for cfg in cfgs])
                
                fw.create_dataset(f'conn_{g}+/{tf}',data=tu+td)
                fw.create_dataset(f'conn_{g}-/{tf}',data=tu-td)
                
    path=f'/p/project1/ngff/li47/code/projectData/05_moments/{ens}/data_merge/conn_2pt.h5'
    with h5py.File(path) as f:
        moms=[list(mom) for mom in f['moms'][:]]
        ind=moms.index([0,0,0])
        for tf in f['data'].keys():
            t=f['data'][tf][:,:,ind]
            tf=int(tf)
            fw.create_dataset(f'conn_c2pt/{tf}',data=t)
    
    path=f'/p/project1/ngff/li47/code/projectData/05_moments/{ens}/data_merge/disc_2pt.h5'
    with h5py.File(path) as f:
        moms=[list(mom) for mom in f['moms'][:]]
        ind=moms.index([0,0,0])
        t=f['data/N_N'][:,:,ind]
        fw.create_dataset(f'disc_c2pt',data=t)
        
    path=f'/p/project1/ngff/li47/code/scratch/run/05_moments_run5_local/{ens}/data_merge/disc_0,0,0,0,0,0.h5'
    with h5py.File(path) as f:
        moms=[list(mom) for mom in f['moms'][:]]
        ind=moms.index([0,0,0,0,0,0])
        
        projs=['P0', 'Px', 'Py', 'Pz']
        inserts=[insert.decode() for insert in f['inserts'][:]]
        
        for g,proj,gm,factor in zip(['gS','gA','gT'],['P0','Pz','Pz'],['id','g5gz','sgmxy'],[1,1j,-1j]):
            for key in f['data'].keys():
                j,tf=key.split('_')
                fla=j[1]; tf=int(tf)
                
                t=f[f'data/{key}'][:]
                t=t[:,:,0,projs.index(proj),inserts.index(gm)]
                t=np.real(t*factor)
                
                fw.create_dataset(f'disc_{g}{fla}/{tf}',data=t)
                
    path='/p/project1/ngff/li47/code/projectData/05_moments/cB211.072.64/data_merge/disc_jvev_local.h5'
    with h5py.File(path) as f:
        for key in f.keys():
            fla=key[1]
            fw.create_dataset(f'vev_gS{fla}',data=f[key][:,0])